# Experiment: A-3 Bollinger Outlier Review

Objective:
- Review the updated A-3 outlier filter based on trailing moving average + Bollinger band.
- Check whether isolated spikes stay flagged while sustained level shifts are restored as trend changes.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd() / ".."
PREPROCESSED_PLUS_DIR = PROJECT_ROOT / "data" / "preprocessed_plus"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

OUTLIER_PATH = PREPROCESSED_PLUS_DIR / "snapshot_outliers.parquet"
MARKET_PRICE_PATH = PREPROCESSED_PLUS_DIR / "snapshot_complex_market_price.parquet"

outliers = pd.read_parquet(OUTLIER_PATH)
market_price = pd.read_parquet(MARKET_PRICE_PATH)
outliers["month"] = pd.to_datetime(outliers["month"], errors="coerce")
if "ref_month" in outliers.columns:
    outliers["ref_month"] = pd.to_datetime(outliers["ref_month"], errors="coerce")
market_price["month"] = pd.to_datetime(market_price["month"], errors="coerce")

trade_files = sorted(PROCESSED_DIR.glob("apt_trade_*.parquet"))
len(outliers), len(market_price), len(trade_files)


## Summary

- `reference_type = moving_average_band`: trailing moving average band 밖에서 끝난 거래
- `reference_type = trend_month_robust_band`: 추세 전환으로 인정된 월 안에서도 월내 중앙값에서 다시 튄 거래


In [ ]:
summary = pd.Series(
    {
        "outlier_rows": len(outliers),
        "unique_groups": outliers[["aptSeq", "area_repr"]].drop_duplicates().shape[0],
        "moving_average_band_rows": int((outliers["reference_type"] == "moving_average_band").sum()),
        "trend_month_robust_rows": int((outliers["reference_type"] == "trend_month_robust_band").sum()),
        "high_outliers": int((outliers["outlier_direction"] == "고가이상치").sum()),
        "low_outliers": int((outliers["outlier_direction"] == "저가이상치").sum()),
    }
)
display(summary.to_frame("value"))

outliers.assign(year=outliers["month"].dt.year).groupby(["year", "reference_type"]).size().unstack(fill_value=0).tail(12)


In [ ]:
top_cases = (
    outliers.assign(abs_dev=outliers["price_deviation_pct"].abs())
    .sort_values(["abs_dev", "month"], ascending=[False, True])
    [[
        "month",
        "aptSeq",
        "apt_name",
        "dong",
        "area_repr",
        "price_per_m2",
        "ref_price",
        "band_width_pct",
        "price_deviation_pct",
        "reference_type",
        "trend_support_months",
        "trend_total_trades",
    ]]
    .head(30)
)
display(top_cases)


## Drill-down Helper

Use `review_case(row_index)` to inspect one flagged case and its surrounding trade history.


In [ ]:
def load_trade_history(apt_seq: str, area_repr: int) -> pd.DataFrame:
    columns = [
        "date",
        "price",
        "price_per_m2",
        "area",
        "area_repr",
        "floor",
        "dong",
        "dong_repr",
        "apt_name",
        "aptSeq",
    ]
    frames: list[pd.DataFrame] = []
    for path in trade_files:
        chunk = pd.read_parquet(path, columns=columns)
        mask = chunk["aptSeq"].eq(apt_seq) & chunk["area_repr"].eq(area_repr)
        if mask.any():
            frames.append(chunk.loc[mask].copy())
    if not frames:
        return pd.DataFrame(columns=columns + ["month"])
    history = pd.concat(frames, ignore_index=True)
    history["date"] = pd.to_datetime(history["date"], errors="coerce")
    history["month"] = history["date"].dt.to_period("M").dt.to_timestamp()
    return history.sort_values(["date", "price_per_m2"]).reset_index(drop=True)


def review_case(row_index: int) -> tuple[pd.Series, pd.DataFrame, pd.DataFrame]:
    case = outliers.iloc[row_index].copy()
    history = load_trade_history(case["aptSeq"], int(case["area_repr"]))
    price_path = market_price[
        (market_price["aptSeq"] == case["aptSeq"])
        & (market_price["area_repr"] == case["area_repr"])
    ].copy()
    window = history[
        history["month"].between(case["month"] - pd.DateOffset(months=6), case["month"] + pd.DateOffset(months=6))
    ].copy()
    display(case.to_frame().T)
    display(window)
    display(price_path.tail(24))
    return case, window, price_path


In [ ]:
# Example: inspect the most extreme remaining outlier
review_case(3)
